In [1]:
# inlegalbert_dual_kg_rag_adaptive_scl_mixup.py
#
# ══════════════════════════════════════════════════════════════════════════════
# BASE  : inlegalbert_dual_kg_rag_adaptive.py
#         (Dual KG-RAG  G_all + G_min  +  Adaptive Confusion Edges)
#
# ★ NEW — Supervised Contrastive Learning (SCL)
#         [ported from inlegalbert_kg_rag_scl_rrc.py]
#   • SupConLoss          — per-class inverse-freq weighted contrastive loss
#   • BalancedContrastiveSampler — minority oversampling for SCL batches
#   • compute_scl_class_weights() — normalised inverse-frequency weights
#   • scl_proj head in InLegalBERT_BiLSTM_MHA_CRF  (Phase A)
#   • fused_scl_proj in DualKGAugmentedModel        (Phase B)
#   • SCL on sent_vecs (Phase A) and fused_sent_vecs (Phase B)
#   • gated by self.training  — zero overhead at inference
#
# ★ NEW — GPU Manifold Mixup (3-Strategy)
#         [ported from inlegalbert_kg_rag_mixup_v4.py]
#   • GPUManifoldMixup
#       S1 IntraRare     — densifies rare-class embedding manifold
#       S2 KG-Guided     — mixes query with its DualKGRetriever neighbours
#       S3 Rare↔HardMaj  — sharpens rare / majority decision boundary
#   • SoftLabelCrossEntropy — soft-label CE for all Mixup targets
#   • FocalLoss             — per-class gamma focal for base training
#   • build_weighted_sampler() — rare-document oversampling (WeightedRandomSampler)
#   • HardExampleBuffer     — hard-example replay in Phase B
#   • compute_focal_class_weights() — capped inverse-freq weights for FocalLoss
#
# ALL Dual KG-RAG + Adaptive Confusion Edge components are UNCHANGED.
# ══════════════════════════════════════════════════════════════════════════════

import os, json, random, time, math
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH = "dataset/build_train.jsonl"
DEV_PATH   = "dataset/build_dev.jsonl"
TEST_PATH  = "dataset/build_test.jsonl"
OUT_DIR    = "rrc_dual_kg_adaptive_scl_mixup_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32
BATCH_DOCS      = 2
NUM_EPOCHS_BASE = 60
NUM_EPOCHS_KG   = 20
BERT_LR         = 1e-5
HEAD_LR         = 5e-4
WEIGHT_DECAY    = 0.05
GRAD_CLIP       = 1.0
DROPOUT         = 0.4

BERT_FREEZE_LAYERS  = 8
BERT_LR_DECAY       = 0.9

SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 2
MHA_HEADS        = 4
MHA_DROPOUT      = 0.1
CTX_LSTM_HIDDEN  = 64
CTX_LSTM_LAYERS  = 2

AUX_CE_WEIGHT    = 0.2
LABEL_SMOOTHING  = 0.1
ES_PATIENCE      = 10
ES_MIN_DELTA     = 1e-4
WARMUP_RATIO     = 0.05
GRADIENT_ACCUMULATION_STEPS = 2
RARE_THRESHOLD   = 0.05

# ── Dual KG-RAG ───────────────────────────────────────────
KG_TOP_K            = 3
KG_TOP_NODES        = 5
KG_MIN_NODES        = 3
KG_HOP              = 1
UNCERTAINTY_THRESH  = 0.7
RARE_ALWAYS_KG      = True
KG_FUSION_DIM       = 256

RARITY_BETA         = 0.3
ALPHA_LOW_ENTROPY   = 0.7
ALPHA_HIGH_ENTROPY  = 0.4

RST_INTRA_THRESH    = 0.6
RST_CROSS_THRESH    = 0.5

# ── Adaptive Confusion Edges ──────────────────────────────
CONF_BASE_ALPHA            = 0.40
CONF_EDGE_WEIGHT_DEFAULT   = 0.90
CONF_TOP_K                 = 3
CONF_SIM_FLOOR             = 0.30
CONF_MAX_PAIRS_PER_CLASS   = 30

# ── SCL [NEW] ─────────────────────────────────────────────
SCL_WEIGHT            = 0.30
SCL_TEMPERATURE       = 0.07
SCL_BASE_TEMPERATURE  = 0.07
SCL_MINORITY_BOOST    = 2.0
SCL_MIN_POSITIVES     = 1
SCL_SAMPLES_PER_CLASS = 4
SCL_PROJ_DIM          = 128

# ── Focal Loss + Oversampling [NEW] ───────────────────────
FOCAL_GAMMA_RARE      = 2.5
FOCAL_GAMMA_MAJ       = 1.0
OVERSAMPLE_RARE_RATIO = 3.0

# ── Hard Example Replay [NEW] ─────────────────────────────
HARD_BUFFER_SIZE        = 300
HARD_REPLAY_FREQ        = 4
HARD_REPLAY_LOSS_THRESH = 0.8
HARD_REPLAY_WEIGHT      = 0.5

# ── GPU Manifold Mixup [NEW] ──────────────────────────────
MIXUP_ALPHA              = 0.40
MIXUP_LOSS_WEIGHT        = 0.35
MIXUP_MIN_RARE_IN_BATCH  = 2

KG_MIXUP_ENABLED         = True
KG_MIXUP_ALPHA           = 0.30
KG_MIXUP_WEIGHT          = 0.25
KG_MIXUP_MAX_PER_BATCH   = 8    # cap rare sents for S2 (limits CPU↔GPU cost)

INTER_MIXUP_ENABLED      = True
INTER_MIXUP_ALPHA        = 0.15
INTER_MIXUP_WEIGHT       = 0.20
INTER_MIXUP_RARE_RATIO   = 0.85

INTRA_RARE_MIXUP_ENABLED = True
INTRA_RARE_MIXUP_WEIGHT  = 0.20

MIXUP_SOFT_TEMP          = 0.50

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=ES_PATIENCE, min_delta=ES_MIN_DELTA):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -1.0
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# ★ FOCAL LOSS  [from mixup file]
# ═══════════════════════════════════════════════════════════
class FocalLoss(nn.Module):
    """
    Per-class gamma focal loss.  Rare classes use FOCAL_GAMMA_RARE;
    majority classes use FOCAL_GAMMA_MAJ.  Combined with inverse-freq
    class weights so rare-class gradients are amplified.
    """
    def __init__(self, class_weights: torch.Tensor, rare_ids: list,
                 gamma_rare: float = FOCAL_GAMMA_RARE,
                 gamma_maj:  float = FOCAL_GAMMA_MAJ,
                 ignore_index: int = -100):
        super().__init__()
        self.ignore_index = ignore_index
        rare_set = set(rare_ids)
        gamma_per_class = torch.full((NUM_LABELS,), gamma_maj)
        for r in rare_set:
            gamma_per_class[r] = gamma_rare
        self.register_buffer("class_weights",   class_weights.float())
        self.register_buffer("gamma_per_class", gamma_per_class)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        valid = targets != self.ignore_index
        if not valid.any():
            return logits.sum() * 0.0
        logits_v  = logits[valid]
        targets_v = targets[valid]
        log_p  = F.log_softmax(logits_v, dim=-1)
        p      = log_p.exp()
        p_t    = p.gather(1, targets_v.unsqueeze(1)).squeeze(1)
        gamma_t  = self.gamma_per_class[targets_v]
        focal_w  = (1.0 - p_t.detach()).pow(gamma_t)
        class_w  = self.class_weights[targets_v]
        ce       = F.nll_loss(log_p, targets_v, reduction="none")
        return (focal_w * class_w * ce).mean()


def compute_focal_class_weights(label_freqs: dict) -> torch.Tensor:
    """Inverse-frequency weights for FocalLoss, capped at 10."""
    weights = torch.ones(NUM_LABELS)
    freqs   = [label_freqs.get(id2label[i], 1e-6) for i in range(NUM_LABELS)]
    inv     = [1.0 / max(f, 1e-6) for f in freqs]
    inv_sum = sum(inv)
    for i, w in enumerate(inv):
        weights[i] = min(w / inv_sum * NUM_LABELS, 10.0)
    return weights


# ═══════════════════════════════════════════════════════════
# ★ SUPERVISED CONTRASTIVE LEARNING  [from SCL file]
# ═══════════════════════════════════════════════════════════
class SupConLoss(nn.Module):
    """
    Supervised Contrastive Loss (Khosla et al., 2020) with per-class
    inverse-frequency weighting and extra minority-class boost.

    Why this beats standard CE for minority classes:
    • CE gradient ∝ prediction error — minority gets tiny total gradient.
    • SCL gradient depends on pairwise similarity — every minority sample
      is compared against ALL others. With w_i boost the minority gradient
      is amplified, forcing tight and well-separated rare-class clusters.
    • Improved embeddings directly benefit KG retrieval quality.
    """
    def __init__(self, temperature: float = SCL_TEMPERATURE,
                 base_temperature: float = SCL_BASE_TEMPERATURE,
                 minority_boost: float   = SCL_MINORITY_BOOST):
        super().__init__()
        self.temperature      = temperature
        self.base_temperature = base_temperature
        self.minority_boost   = minority_boost

    def forward(self, features: torch.Tensor,   # (N, D) L2-normalised
                labels:        torch.Tensor,    # (N,)   int class ids
                class_weights: torch.Tensor,    # (C,)   per-class weights
                rare_ids: list = None) -> torch.Tensor:
        device = features.device
        N = features.shape[0]
        if N < 2:
            return torch.tensor(0.0, device=device, requires_grad=True)

        # Per-sample weights with optional minority boost
        sample_weights = class_weights[labels]
        if rare_ids:
            rare_mask = torch.zeros(N, dtype=torch.bool, device=device)
            for rid in rare_ids:
                rare_mask |= (labels == rid)
            sample_weights = sample_weights.clone()
            sample_weights[rare_mask] *= self.minority_boost

        # Cosine similarity matrix (features are already L2-normalised)
        sim_matrix = torch.mm(features, features.T) / self.temperature

        # Positive / negative masks
        labels_col = labels.unsqueeze(0)
        labels_row = labels.unsqueeze(1)
        pos_mask   = (labels_row == labels_col).float()
        self_mask  = torch.eye(N, device=device)
        pos_mask   = pos_mask - self_mask

        n_pos_per_anchor = pos_mask.sum(dim=1)
        valid = n_pos_per_anchor >= SCL_MIN_POSITIVES
        if not valid.any():
            return torch.tensor(0.0, device=device, requires_grad=True)

        logits_mask  = 1.0 - self_mask
        exp_sim      = torch.exp(
            sim_matrix - sim_matrix.max(dim=1, keepdim=True).values
        ) * logits_mask

        log_prob = sim_matrix - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-9)
        mean_log_prob_pos = (pos_mask * log_prob).sum(dim=1) / (
            n_pos_per_anchor.clamp(min=1))

        loss_per_anchor = (-(self.temperature / self.base_temperature)
                           * mean_log_prob_pos * sample_weights)
        return loss_per_anchor[valid].mean()


class BalancedContrastiveSampler:
    """
    For each document batch, return a class-balanced subset of embeddings
    for SCL computation.  Minority classes are oversampled with small
    Gaussian noise (feature-space synthetic positives).
    """
    def __init__(self, samples_per_class: int = SCL_SAMPLES_PER_CLASS,
                 noise_std: float = 0.01,
                 rare_ids:  list  = None):
        self.samples_per_class = samples_per_class
        self.noise_std         = noise_std
        self.rare_ids          = set(rare_ids or [])

    def sample(self, embeddings: torch.Tensor,
               labels: torch.Tensor):
        device    = embeddings.device
        emb_list, lbl_list = [], []
        for lbl in labels.unique().tolist():
            idx  = (labels == lbl).nonzero(as_tuple=True)[0]
            embs = embeddings[idx]
            k    = embs.shape[0]
            if k >= self.samples_per_class:
                chosen = torch.randperm(k, device=device)[:self.samples_per_class]
                emb_list.append(embs[chosen])
            else:
                emb_list.append(embs)
                n_needed = self.samples_per_class - k
                if int(lbl) in self.rare_ids or k < self.samples_per_class:
                    reps  = embs[torch.randint(0, k, (n_needed,), device=device)]
                    noise = torch.randn_like(reps) * self.noise_std
                    emb_list.append(reps + noise)
                    k = self.samples_per_class
            n_added = min(k, self.samples_per_class)
            lbl_list.append(torch.full((n_added,), lbl,
                                       dtype=torch.long, device=device))
        if not emb_list:
            return embeddings, labels
        out_embs = torch.cat(emb_list, dim=0)
        out_labs = torch.cat(lbl_list, dim=0)
        perm = torch.randperm(out_embs.shape[0], device=device)
        return out_embs[perm], out_labs[perm]


def compute_scl_class_weights(label_freqs: dict, rare_ids: list,
                               smoothing: float = 0.1) -> torch.Tensor:
    """Normalised inverse-frequency weights for SCL (mean = 1)."""
    weights = []
    for i in range(NUM_LABELS):
        freq = label_freqs.get(id2label[i], 0.0)
        weights.append(1.0 / (freq + smoothing))
    weights = torch.tensor(weights, dtype=torch.float)
    weights = weights / weights.mean()
    print("\n⚖️  SCL class weights (inverse freq, normalised to mean=1):")
    for i, lbl in enumerate(LABELS):
        flag = " ← RARE" if label2id[lbl] in rare_ids else ""
        print(f"   {lbl:<20}  weight={float(weights[i]):.3f}{flag}")
    return weights


# ═══════════════════════════════════════════════════════════
# ★ GPU MANIFOLD MIXUP  [from mixup file]
# ═══════════════════════════════════════════════════════════
class GPUManifoldMixup(nn.Module):
    """
    All-GPU Manifold Mixup — three strategies:
      S1 IntraRare   : mix two rare embeddings from the same batch
      S2 KG-Guided   : mix a rare query with its top-1 KG neighbour
                       (called externally with pre-retrieved neighbours)
      S3 Inter-class : mix rare with hard-majority (confused class)

    All lambda values are sampled from Beta(α,α) on GPU.
    Returns mixed embeddings + soft blended one-hot targets.
    """
    def __init__(self, num_labels: int = NUM_LABELS,
                 alpha:       float = MIXUP_ALPHA,
                 kg_alpha:    float = KG_MIXUP_ALPHA,
                 inter_alpha: float = INTER_MIXUP_ALPHA,
                 rare_ratio:  float = INTER_MIXUP_RARE_RATIO,
                 soft_temp:   float = MIXUP_SOFT_TEMP):
        super().__init__()
        self.num_labels  = num_labels
        self.alpha       = alpha
        self.kg_alpha    = kg_alpha
        self.inter_alpha = inter_alpha
        self.rare_ratio  = rare_ratio
        self.soft_temp   = soft_temp

    @staticmethod
    def _beta_gpu(alpha: float, size: int, device) -> torch.Tensor:
        if alpha <= 0:
            return torch.ones(size, device=device)
        dist = torch.distributions.Beta(
            torch.tensor(alpha, device=device),
            torch.tensor(alpha, device=device))
        return dist.sample((size,))

    @staticmethod
    def _onehot(labels: torch.Tensor, num_classes: int) -> torch.Tensor:
        return F.one_hot(labels.clamp(min=0), num_classes).float()

    # ── S1: Intra-Rare Mixup ─────────────────────────────
    def intra_rare_mixup(self, embs: torch.Tensor, labels: torch.Tensor,
                         rare_ids_t: torch.Tensor):
        """
        For each rare sentence, pick another rare sentence at random and
        interpolate.  Densifies the rare-class manifold.
        rare_ids_t : 1-D GPU tensor of rare label ids.
        Returns (mixed_embs, soft_labels) or (None, None).
        """
        is_rare  = (labels.unsqueeze(1) == rare_ids_t.unsqueeze(0)).any(1)
        rare_idx = is_rare.nonzero(as_tuple=True)[0]
        if rare_idx.shape[0] < MIXUP_MIN_RARE_IN_BATCH:
            return None, None

        N_r  = rare_idx.shape[0]
        lam  = self._beta_gpu(self.alpha, N_r, embs.device).clamp(0.1, 0.9)
        perm = torch.randperm(N_r, device=embs.device)
        rare2 = rare_idx[perm]

        lam_e = lam.unsqueeze(1)
        mixed = lam_e * embs[rare_idx] + (1.0 - lam_e) * embs[rare2]
        oh1   = self._onehot(labels[rare_idx],  self.num_labels)
        oh2   = self._onehot(labels[rare2],     self.num_labels)
        soft  = lam_e * oh1 + (1.0 - lam_e) * oh2
        return mixed, soft

    # ── S2: KG-Guided Mixup (list-based neighbour interface) ──
    def kg_guided_mixup_from_list(self, h_i: torch.Tensor,
                                   label_id: int,
                                   neighbours: list) -> tuple:
        """
        Adapted for File-1's list-based retriever.
        neighbours : list of (Tensor(D,), float) from MinorityAwareRetriever.
        Returns (mixed_emb (D,), soft_label (C,)) or (None, None).
        """
        if not neighbours:
            return None, None
        device = h_i.device
        nb_embs = torch.stack([nb[0] for nb in neighbours]).to(device)
        nb_w    = torch.tensor([nb[1] for nb in neighbours],
                               device=device, dtype=torch.float)
        top_idx = nb_w.argmax()
        top_emb = nb_embs[top_idx]

        lam = float(self._beta_gpu(self.kg_alpha, 1, device)
                    .clamp(0.6, 0.9).item())
        mixed = lam * h_i + (1.0 - lam) * top_emb
        oh    = self._onehot(
            torch.tensor([label_id], device=device), self.num_labels
        ).squeeze(0)
        soft  = lam * oh + (1.0 - lam) / self.num_labels
        return mixed, soft

    # ── S3: Rare ↔ Hard-Majority Inter-class Mixup ────────
    def inter_class_mixup(self, embs: torch.Tensor, labels: torch.Tensor,
                          rare_ids_t: torch.Tensor,
                          conf_maj_ids_t: torch.Tensor):
        """
        For each rare sentence, find a hard-majority (confused class)
        sentence and mix.  Rare label dominates via rare_ratio weight.
        Returns (mixed_embs, soft_labels) or (None, None).
        """
        is_rare  = (labels.unsqueeze(1) == rare_ids_t.unsqueeze(0)).any(1)
        rare_idx = is_rare.nonzero(as_tuple=True)[0]
        if rare_idx.shape[0] < 2:
            return None, None

        is_hard_maj = (
            (labels.unsqueeze(1) == conf_maj_ids_t.unsqueeze(0)).any(1) & ~is_rare
        )
        maj_idx = is_hard_maj.nonzero(as_tuple=True)[0]
        if maj_idx.shape[0] < 1:
            maj_idx = (~is_rare).nonzero(as_tuple=True)[0]
        if maj_idx.shape[0] < 1:
            return None, None

        N_r  = rare_idx.shape[0]
        lam  = self._beta_gpu(
            self.inter_alpha, N_r, embs.device
        ).clamp(0.0, self.inter_alpha * 2)

        perm_idx = torch.randint(0, maj_idx.shape[0], (N_r,), device=embs.device)
        maj_part = maj_idx[perm_idx]

        lam_e = lam.unsqueeze(1)
        mixed = (1.0 - lam_e) * embs[rare_idx] + lam_e * embs[maj_part]
        oh1   = self._onehot(labels[rare_idx],  self.num_labels)
        oh2   = self._onehot(labels[maj_part],   self.num_labels)
        soft  = self.rare_ratio * oh1 + (1.0 - self.rare_ratio) * oh2
        return mixed, soft


class SoftLabelCrossEntropy(nn.Module):
    """Cross-entropy with soft (blended one-hot) targets for Mixup."""
    def __init__(self, temperature: float = MIXUP_SOFT_TEMP,
                 class_weights: torch.Tensor = None):
        super().__init__()
        self.temperature = temperature
        if class_weights is not None:
            self.register_buffer("class_weights", class_weights.float())
        else:
            self.class_weights = None

    def forward(self, logits: torch.Tensor,
                soft_labels: torch.Tensor) -> torch.Tensor:
        log_p = F.log_softmax(logits / self.temperature, dim=-1)
        if self.class_weights is not None:
            dom_cls = soft_labels.argmax(dim=-1)
            w       = self.class_weights[dom_cls]
            loss    = -(soft_labels * log_p).sum(dim=-1)
            return (loss * w).mean()
        return -(soft_labels * log_p).sum(dim=-1).mean()


# ═══════════════════════════════════════════════════════════
# ★ HARD EXAMPLE REPLAY BUFFER  [from mixup file]
# ═══════════════════════════════════════════════════════════
class HardExampleBuffer:
    """Stores high-loss batches containing rare-class sentences."""
    def __init__(self, max_size: int = HARD_BUFFER_SIZE):
        self.max_size = max_size
        self.buf: list = []

    def update(self, loss_val: float, batch_cpu: tuple):
        self.buf.append((loss_val, batch_cpu))
        if len(self.buf) > self.max_size:
            self.buf.sort(key=lambda x: -x[0])
            self.buf = self.buf[:self.max_size // 2]

    def sample(self):
        if not self.buf:
            return None
        losses = torch.tensor([x[0] for x in self.buf], dtype=torch.float)
        probs  = F.softmax(losses, dim=0)
        idx    = int(torch.multinomial(probs, 1).item())
        return self.buf[idx][1]

    def __len__(self):
        return len(self.buf)


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))
        if not sents or len(sents) != len(labs):
            continue
        all_docs.append((sents[:max_sents], labs[:max_sents]))
    return all_docs


def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}
    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]
    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        count = counts.get(label2id[lbl], 0)
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({count:5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, label_freqs


# ═══════════════════════════════════════════════════════════
# DATASET  +  ★ WEIGHTED SAMPLER  [sampler from mixup file]
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get("token_type_ids",
                                      torch.zeros_like(enc["input_ids"])),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def build_weighted_sampler(docs: list, rare_ids: list,
                            oversample_ratio: float = OVERSAMPLE_RARE_RATIO
                            ) -> WeightedRandomSampler:
    """Over-sample documents that contain at least one rare-class sentence."""
    rare_set = set(rare_ids)
    weights  = [oversample_ratio if any(l in rare_set for l in labs)
                else 1.0
                for _, labs in docs]
    w_tensor = torch.tensor(weights, dtype=torch.float)
    return WeightedRandomSampler(w_tensor, num_samples=len(w_tensor),
                                  replacement=True)


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L     = batch[0]["input_ids"].shape[1]
    B     = len(batch)
    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)
    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t
    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# MULTI-HEAD ATTENTION POOLING
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads
        self.query = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)
        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)
        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x: torch.Tensor,
                key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        N, L, H = x.shape
        K = self.key_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.val_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        attn = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        if key_padding_mask is not None:
            attn = attn.masked_fill(
                key_padding_mask.unsqueeze(1).unsqueeze(2), -1e9)
        attn = self.attn_drop(F.softmax(attn, dim=-1))
        return self.out_proj(torch.matmul(attn, V).squeeze(2).reshape(N, H))


# ═══════════════════════════════════════════════════════════
# BASE MODEL  (+ ★ SCL projection head)
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_CRF(nn.Module):
    """
    Original Dual KG-RAG base model extended with:
      • self.scl_proj  — 2-layer MLP projection head for SCL
        Maps sent_out_dim (256) → SCL_PROJ_DIM (128), L2-normalised.
        Used only during training (self.training=True).
        Discarded at inference (standard SimCLR / SupCon practice).
      • get_scl_embeddings() — projects + normalises sent_vecs.
      • SCL loss is computed internally in forward() when
        _scl_loss_fn is set AND self.training is True.
    """
    def __init__(
        self,
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
        scl_proj_dim     = SCL_PROJ_DIM,          # ★ SCL
        # SCL components (optional — set None to disable)
        scl_loss_fn      = None,
        scl_class_weights: torch.Tensor = None,
        scl_sampler      = None,
        rare_ids: list   = None,
    ):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size
        self.dropout  = nn.Dropout(dropout)
        self._freeze_bert_layers(freeze_layers)

        self.sent_bilstm = nn.LSTM(
            input_size=self.bert_dim, hidden_size=sent_lstm_hidden,
            num_layers=sent_lstm_layers, bidirectional=True, batch_first=True,
            dropout=dropout if sent_lstm_layers > 1 else 0.0)
        self.sent_out_dim    = sent_lstm_hidden * 2   # 256
        self.mha_pooling     = MultiHeadAttentionPooling(
            self.sent_out_dim, mha_heads, mha_dropout)
        self.sent_layer_norm = nn.LayerNorm(self.sent_out_dim)

        self.ctx_bilstm = nn.LSTM(
            input_size=self.sent_out_dim, hidden_size=ctx_lstm_hidden,
            num_layers=ctx_lstm_layers, bidirectional=True, batch_first=True,
            dropout=dropout if ctx_lstm_layers > 1 else 0.0)
        self.ctx_out_dim = ctx_lstm_hidden * 2   # 128

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim, self.ctx_out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim // 2, num_labels))

        self.crf    = CRF(num_tags=num_labels, batch_first=True)
        self.ce_loss = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING, ignore_index=-100)

        # ★ SCL projection head (training only)
        self.scl_proj_dim = scl_proj_dim
        self.scl_proj = nn.Sequential(
            nn.Linear(self.sent_out_dim, self.sent_out_dim),
            nn.GELU(),
            nn.Linear(self.sent_out_dim, scl_proj_dim),
        )

        # Store SCL components (gated by self.training inside forward)
        self._scl_loss_fn       = scl_loss_fn
        self._scl_class_weights = scl_class_weights
        self._scl_sampler       = scl_sampler
        self._rare_ids_list     = rare_ids or []
        self._scl_enabled       = (scl_loss_fn is not None and
                                   scl_class_weights is not None)

    def _freeze_bert_layers(self, n_freeze: int):
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False
        enc = self.bert.encoder.layer
        for i in range(min(n_freeze, len(enc))):
            for param in enc[i].parameters():
                param.requires_grad = False
        n_total = len(enc)
        print(f"\n❄️  BERT frozen: embeddings + layers 0-{n_freeze-1}.")
        print(f"🔥 BERT trainable: layers {n_freeze}-{n_total-1} + pooler.\n")

    # ── Sentence encoding (unchanged) ────────────────────
    def encode_sentences(self, input_ids, attention_mask, token_type_ids,
                         lengths=None):
        B, T, L = input_ids.shape
        N = B * T
        flat_ids   = input_ids.view(N, L)
        flat_mask  = attention_mask.view(N, L)
        flat_types = token_type_ids.view(N, L)
        valid = flat_mask.sum(dim=-1) > 0
        token_embs = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)
        if valid.any():
            out = self.bert(
                input_ids=flat_ids[valid],
                attention_mask=flat_mask[valid],
                token_type_ids=flat_types[valid])
            token_embs[valid] = out.last_hidden_state.to(token_embs.dtype)
        token_embs = self.dropout(token_embs)
        lstm_out, _ = self.sent_bilstm(token_embs)
        lstm_out    = self.dropout(lstm_out)
        pad_mask = (flat_mask == 0).clone()
        pad_mask[~valid] = False
        sent_vecs = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs = self.sent_layer_norm(sent_vecs)
        sent_vecs = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)
        return sent_vecs.view(B, T, -1)

    # ── ★ SCL projection ─────────────────────────────────
    def get_scl_embeddings(self, sent_vecs: torch.Tensor,
                           labels: torch.Tensor):
        """
        Projects sent_vecs through scl_proj and L2-normalises.
        Returns (proj (N, scl_proj_dim), flat_labels (N,))
        for valid (non-padding) sentences only.
        """
        B, T, D = sent_vecs.shape
        flat_vecs   = sent_vecs.reshape(B * T, D)
        flat_labels = labels.reshape(B * T)
        valid_mask  = flat_labels != -100
        flat_vecs   = flat_vecs[valid_mask]
        flat_labels = flat_labels[valid_mask]
        if flat_vecs.shape[0] == 0:
            return None, None
        proj = self.scl_proj(flat_vecs)
        proj = F.normalize(proj, dim=-1)
        return proj, flat_labels

    # ── Emissions (unchanged) ─────────────────────────────
    def get_emissions(self, input_ids, attention_mask, token_type_ids,
                      lengths=None):
        sent_vecs = self.encode_sentences(
            input_ids, attention_mask, token_type_ids, lengths=lengths)
        sent_vecs_drop = self.dropout(sent_vecs)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sent_vecs_drop, lengths.cpu(),
                batch_first=True, enforce_sorted=False)
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _ = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True)
        else:
            ctx_out, _ = self.ctx_bilstm(sent_vecs_drop)
        ctx_out   = self.dropout(ctx_out)
        emissions = self.classifier(ctx_out)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)
        return sent_vecs, ctx_out, emissions

    # ── Forward (+ ★ SCL) ─────────────────────────────────
    def forward(self, input_ids, attention_mask, token_type_ids,
                labels=None, lengths=None):
        sent_vecs, _, emissions = self.get_emissions(
            input_ids, attention_mask, token_type_ids, lengths=lengths)

        if lengths is not None:
            B, T, _ = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool,
                              device=emissions.device)

        if labels is not None:
            safe = labels.clone(); safe[safe == -100] = 0
            crf_loss = -self.crf(emissions, safe, mask=mask, reduction="mean")
            B2, T2, C = emissions.shape
            ce_loss = self.ce_loss(
                emissions.reshape(B2 * T2, C), labels.reshape(B2 * T2))

            total_loss = crf_loss + AUX_CE_WEIGHT * ce_loss

            # ★ SCL (training only, if components are configured)
            if self.training and self._scl_enabled:
                proj_embs, flat_labels = self.get_scl_embeddings(sent_vecs, labels)
                if proj_embs is not None and proj_embs.shape[0] >= 2:
                    if self._scl_sampler is not None:
                        proj_embs, flat_labels = self._scl_sampler.sample(
                            proj_embs, flat_labels)
                    cw = self._scl_class_weights.to(emissions.device)
                    scl_loss = self._scl_loss_fn(
                        proj_embs, flat_labels, cw,
                        rare_ids=self._rare_ids_list)
                    total_loss = total_loss + SCL_WEIGHT * scl_loss

            return total_loss, emissions
        else:
            return self.crf.decode(emissions, mask=mask), emissions


# ═══════════════════════════════════════════════════════════
# ★ CONFUSION ANALYSIS & ADAPTIVE WEIGHT FUNCTIONS
# ═══════════════════════════════════════════════════════════
@torch.no_grad()
def compute_confusion_pairs_and_matrix(
        model, dataset, rare_ids, device=DEVICE, top_k=CONF_TOP_K) -> tuple:
    """
    Run base model on training set → collect:
      (a) confusion_pairs : {rare_lid → [top-K confused majority lids]}
      (b) raw_cm          : (NUM_LABELS, NUM_LABELS) int numpy matrix
    """
    model.eval()
    loader   = DataLoader(dataset, batch_size=2, shuffle=False,
                          collate_fn=collate_rrc)
    rare_set = set(rare_ids)
    all_preds, all_trues = [], []

    for ids, attn, ttype, labels, lengths in loader:
        ids   = ids.to(device);    attn    = attn.to(device)
        ttype = ttype.to(device);  lengths = lengths.to(device)
        decoded, _ = model(ids, attn, ttype, labels=None, lengths=lengths)
        for i, seq in enumerate(decoded):
            n = int(lengths[i].item())
            all_preds.extend(seq)
            all_trues.extend(labels[i, :n].tolist())

    from sklearn.metrics import confusion_matrix as sk_cm
    str_t = [id2label[x] for x in all_trues]
    str_p = [id2label[x] for x in all_preds]
    raw_cm = sk_cm(str_t, str_p, labels=LABELS)

    confusion = defaultdict(Counter)
    for true, pred in zip(all_trues, all_preds):
        if true in rare_set and pred != true:
            confusion[true][pred] += 1

    confusion_pairs = {}
    print("\n★ Confusion pair analysis (base model on training set):")
    print(f"  {'Rare class':<22} {'Top confused majority classes'}")
    print("  " + "-" * 70)
    for rid in rare_ids:
        counter = confusion.get(rid, Counter())
        total_errors = sum(counter.values())
        top  = [cls for cls, _ in counter.most_common(top_k)]
        confusion_pairs[rid] = top
        names = [(id2label[c], counter[c]) for c in top]
        name_str = ", ".join(f"{n}({cnt})" for n, cnt in names)
        print(f"  {id2label[rid]:<22} → {name_str}  (errors: {total_errors})")

    total_rare_errors = sum(sum(c.values()) for c in confusion.values())
    total_rare_tokens = sum(all_trues.count(r) for r in rare_ids)
    print(f"\n  Rare tokens: {total_rare_tokens} | "
          f"Rare errors: {total_rare_errors} "
          f"({100*total_rare_errors/max(1,total_rare_tokens):.1f}%)")
    return confusion_pairs, raw_cm


def compute_adaptive_confusion_weights(
        raw_cm: np.ndarray,
        rare_ids: list,
        base_alpha: float = CONF_BASE_ALPHA) -> torch.Tensor:
    """
    W[i,j] = base_alpha + (1-base_alpha) * row_norm(C)[i,j]
    Majority rows keep CONF_EDGE_WEIGHT_DEFAULT.
    """
    W = torch.full((NUM_LABELS, NUM_LABELS),
                   CONF_EDGE_WEIGHT_DEFAULT, dtype=torch.float32)
    print("\n★ Adaptive confusion edge weights (rare class rows):")
    for i in rare_ids:
        row       = raw_cm[i].copy().astype(float)
        row[i]    = 0.0
        row_total = row.sum()
        if row_total < 1.0:
            W[i, :] = base_alpha
            W[i, i] = 0.0
            print(f"  {id2label[i]:<20}  no errors → floor={base_alpha:.3f}")
            continue
        row_norm = row / row_total
        top3_j   = sorted(range(NUM_LABELS), key=lambda j: -row_norm[j])[:3]
        for j in range(NUM_LABELS):
            W[i, j] = 0.0 if j == i else float(
                base_alpha + (1.0 - base_alpha) * row_norm[j])
        top3_str = ", ".join(
            f"{id2label[j]}={float(W[i,j]):.3f}" for j in top3_j if j != i)
        print(f"  {id2label[i]:<20}  top-3: {top3_str}")
    return W


def save_adaptive_weight_heatmap(confusion_weights, rare_ids, out_dir=OUT_DIR):
    rare_labels = [id2label[r] for r in rare_ids]
    w_np = confusion_weights.numpy()
    sub  = w_np[np.ix_(rare_ids, list(range(NUM_LABELS)))]
    fig, ax = plt.subplots(figsize=(14, max(4, len(rare_ids))))
    sns.heatmap(sub, annot=True, fmt=".3f",
                xticklabels=LABELS, yticklabels=rare_labels,
                cmap="YlOrRd", vmin=0.0, vmax=1.0, ax=ax)
    ax.set_title("Adaptive Confusion Edge Weights  W[rare → majority]")
    ax.set_xlabel("Majority class (destination)")
    ax.set_ylabel("Rare class (source)")
    plt.tight_layout()
    path = os.path.join(out_dir, "adaptive_confusion_weights.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"  Saved adaptive weight heatmap → {path}")


# ═══════════════════════════════════════════════════════════
# KNOWLEDGE GRAPH  (unchanged from Dual KG-RAG)
# ═══════════════════════════════════════════════════════════
class KnowledgeGraph:
    def __init__(self, emb_dim=KG_FUSION_DIM, name="global"):
        self.emb_dim       = emb_dim
        self.name          = name
        self.nodes         = defaultdict(list)
        self.intra_edges   = defaultdict(list)
        self.cross_edges   = []
        self.conf_cx_edges = []
        self._stacked      = {}

    def add_nodes(self, embeddings, label_ids, texts=None):
        embs = embeddings.detach().cpu()
        for k, (emb, lid) in enumerate(zip(embs, label_ids)):
            text = texts[k] if texts is not None else ""
            self.nodes[lid].append({"emb": emb, "text": text})
        self._stacked = {}

    def build_edges(self, intra_thresh=RST_INTRA_THRESH,
                    cross_thresh=RST_CROSS_THRESH,
                    max_intra_edges_per_node=5,
                    max_cross_edges=2000):
        print(f"  Building {self.name} KG RST edges ...")
        self._stacked    = {}
        self.intra_edges = defaultdict(list)
        self.cross_edges = []
        for lid, node_list in self.nodes.items():
            N = len(node_list)
            if N < 2:
                continue
            embs      = torch.stack([n["emb"] for n in node_list])
            embs_norm = F.normalize(embs, dim=-1)
            sim_mat   = torch.mm(embs_norm, embs_norm.T)
            for i in range(N - 1):
                w = float(sim_mat[i, i+1].item())
                self.intra_edges[lid].append((i, i+1, max(0.0, w)))
            for i in range(N):
                sims = sim_mat[i].clone()
                sims[max(0, i-1):i+2] = -1
                count = 0
                while count < max_intra_edges_per_node:
                    j = int(sims.argmax().item())
                    if sims[j] < intra_thresh:
                        break
                    self.intra_edges[lid].append((i, j, float(sims[j].item())))
                    sims[j] = -1
                    count  += 1
            self._stacked[lid] = embs
        label_ids   = list(self.nodes.keys())
        cross_count = 0
        for a in range(len(label_ids)):
            if cross_count >= max_cross_edges:
                break
            for b in range(a + 1, len(label_ids)):
                if cross_count >= max_cross_edges:
                    break
                la, lb = label_ids[a], label_ids[b]
                ea = self._get_stacked(la); eb = self._get_stacked(lb)
                if ea is None or eb is None:
                    continue
                sim_mat = torch.mm(F.normalize(ea, dim=-1),
                                   F.normalize(eb, dim=-1).T)
                high = (sim_mat >= cross_thresh).nonzero(as_tuple=False)
                for pair in high[:50]:
                    ni, nj = int(pair[0]), int(pair[1])
                    w = float(sim_mat[ni, nj].item())
                    self.cross_edges.append((la, ni, lb, nj, w))
                    cross_count += 1
        n_intra = sum(len(v) for v in self.intra_edges.values())
        n_nodes = sum(len(v) for v in self.nodes.values())
        print(f"  [{self.name}] {n_nodes} nodes | "
              f"{n_intra} intra-edges | {len(self.cross_edges)} cross-edges")

    def add_confusion_edges(self, confusion_pairs, confusion_weights,
                             sim_floor=CONF_SIM_FLOOR,
                             max_pairs=CONF_MAX_PAIRS_PER_CLASS):
        self.conf_cx_edges = []
        total_added = 0
        print(f"  [{self.name}] Building adaptive confusion cross-edges ...")
        for rare_lid, confused_lids in confusion_pairs.items():
            embs_rare = self._get_stacked(rare_lid)
            if embs_rare is None:
                continue
            for clid in confused_lids:
                embs_conf = self._get_stacked(clid)
                if embs_conf is None:
                    continue
                edge_w  = float(confusion_weights[rare_lid, clid].item())
                nr      = F.normalize(embs_rare, dim=-1)
                nc      = F.normalize(embs_conf, dim=-1)
                sim_mat = torch.mm(nr, nc.T)
                rows, cols = torch.where(sim_mat >= sim_floor)
                if rows.shape[0] == 0:
                    best = sim_mat.argmax()
                    ri   = int(best // sim_mat.shape[1])
                    ci   = int(best %  sim_mat.shape[1])
                    self.conf_cx_edges.append((rare_lid, ri, clid, ci, edge_w))
                    total_added += 1
                    continue
                sims_flat = sim_mat[rows, cols]
                order     = sims_flat.argsort(descending=True)
                rows      = rows[order[:max_pairs]]
                cols      = cols[order[:max_pairs]]
                for ri, ci in zip(rows.tolist(), cols.tolist()):
                    self.conf_cx_edges.append((rare_lid, ri, clid, ci, edge_w))
                total_added += len(rows)
                print(f"    {id2label[rare_lid]:<20} → {id2label[clid]:<20} "
                      f"w={edge_w:.4f}  ({len(rows)} pairs)")
        print(f"  [{self.name}] Confusion edges added: {total_added}")
        return total_added

    def _get_stacked(self, lid):
        if lid not in self._stacked:
            if lid not in self.nodes or not self.nodes[lid]:
                return None
            self._stacked[lid] = torch.stack(
                [n["emb"] for n in self.nodes[lid]])
        return self._stacked[lid]

    def save(self, path):
        data = {
            "name": self.name,
            "nodes": {
                str(k): [{"emb": n["emb"].tolist(), "text": n["text"]}
                          for n in v]
                for k, v in self.nodes.items()
            },
            "intra_edges":  {str(k): v for k, v in self.intra_edges.items()},
            "cross_edges":   self.cross_edges,
            "conf_cx_edges": self.conf_cx_edges,
        }
        with open(path, "w") as f:
            json.dump(data, f)
        print(f"  [{self.name}] KG saved → {path}")

    @classmethod
    def load(cls, path, emb_dim=KG_FUSION_DIM):
        with open(path) as f:
            data = json.load(f)
        kg = cls(emb_dim=emb_dim, name=data.get("name", "global"))
        for k, node_list in data["nodes"].items():
            lid = int(k)
            for n in node_list:
                kg.nodes[lid].append({"emb": torch.tensor(n["emb"]),
                                      "text": n["text"]})
        for k, edges in data["intra_edges"].items():
            kg.intra_edges[int(k)] = [tuple(e) for e in edges]
        kg.cross_edges   = [tuple(e) for e in data.get("cross_edges",   [])]
        kg.conf_cx_edges = [tuple(e) for e in data.get("conf_cx_edges", [])]
        print(f"  [{kg.name}] KG loaded ← {path}  "
              f"(conf_cx_edges: {len(kg.conf_cx_edges)})")
        return kg


# ═══════════════════════════════════════════════════════════
# DUAL KNOWLEDGE GRAPH  (unchanged)
# ═══════════════════════════════════════════════════════════
class DualKnowledgeGraph:
    def __init__(self, rare_ids, emb_dim=KG_FUSION_DIM):
        self.rare_ids = set(rare_ids)
        self.g_all    = KnowledgeGraph(emb_dim=emb_dim, name="global")
        self.g_min    = KnowledgeGraph(emb_dim=emb_dim, name="minority")

    def add_nodes(self, embeddings, label_ids, texts=None):
        self.g_all.add_nodes(embeddings, label_ids, texts)
        min_embs, min_ids, min_texts = [], [], []
        for k, (emb, lid) in enumerate(zip(embeddings, label_ids)):
            if lid in self.rare_ids:
                min_embs.append(emb)
                min_ids.append(lid)
                if texts:
                    min_texts.append(texts[k])
        if min_embs:
            self.g_min.add_nodes(torch.stack(min_embs), min_ids,
                                  min_texts if texts else None)

    def build_edges(self):
        self.g_all.build_edges()
        self.g_min.build_edges(
            intra_thresh=RST_INTRA_THRESH - 0.1,
            cross_thresh=RST_CROSS_THRESH - 0.1)

    def build_confusion_edges(self, confusion_pairs, confusion_weights):
        print("\n★ Inserting adaptive confusion edges into G_all ...")
        n_all = self.g_all.add_confusion_edges(confusion_pairs, confusion_weights)
        print("\n★ Inserting adaptive confusion edges into G_min ...")
        n_min = self.g_min.add_confusion_edges(
            confusion_pairs, confusion_weights,
            sim_floor=CONF_SIM_FLOOR - 0.05,
            max_pairs=CONF_MAX_PAIRS_PER_CLASS)
        print(f"\n  Total confusion edges: G_all={n_all} | G_min={n_min}")

    def save(self, dir_path):
        self.g_all.save(os.path.join(dir_path, "kg_global.json"))
        self.g_min.save(os.path.join(dir_path, "kg_minority.json"))

    @classmethod
    def load(cls, dir_path, rare_ids, emb_dim=KG_FUSION_DIM):
        dual       = cls(rare_ids=rare_ids, emb_dim=emb_dim)
        dual.g_all = KnowledgeGraph.load(
            os.path.join(dir_path, "kg_global.json"), emb_dim=emb_dim)
        dual.g_min = KnowledgeGraph.load(
            os.path.join(dir_path, "kg_minority.json"), emb_dim=emb_dim)
        return dual


# ═══════════════════════════════════════════════════════════
# UNCERTAINTY ESTIMATOR  (unchanged)
# ═══════════════════════════════════════════════════════════
class UncertaintyEstimator:
    def __init__(self, num_classes=NUM_LABELS, threshold=UNCERTAINTY_THRESH):
        self.log_C     = math.log(num_classes)
        self.threshold = threshold

    def entropy(self, logits):
        probs = F.softmax(logits, dim=-1)
        H = -(probs * (probs + 1e-9).log()).sum(dim=-1)
        return H / self.log_C

    def is_uncertain(self, logits):
        return self.entropy(logits) > self.threshold

    def top_label(self, logits):
        return logits.argmax(dim=-1)


# ═══════════════════════════════════════════════════════════
# MINORITY-AWARE RETRIEVER  (unchanged)
# ═══════════════════════════════════════════════════════════
class MinorityAwareRetriever:
    def __init__(self, kg: KnowledgeGraph, label_freqs: dict, rare_ids: list,
                 top_k=KG_TOP_K, top_nodes_k1=KG_TOP_NODES,
                 top_nodes_k2=KG_MIN_NODES, hop=KG_HOP, beta=RARITY_BETA):
        self.kg           = kg
        self.rare_ids     = set(rare_ids)
        self.top_k        = top_k
        self.top_nodes_k1 = top_nodes_k1
        self.top_nodes_k2 = top_nodes_k2
        self.hop          = hop
        self.beta         = beta
        self.rarity       = {}
        for i in range(NUM_LABELS):
            freq = label_freqs.get(id2label[i], 0.0)
            self.rarity[i] = 1.0 / (math.log(freq + 1.0 + 1e-6))
        max_rar = max(self.rarity.values()) or 1.0
        self.rarity = {k: v / max_rar for k, v in self.rarity.items()}
        self._conf_by_rare = defaultdict(list)
        for edge in self.kg.conf_cx_edges:
            self._conf_by_rare[edge[0]].append(edge)

    def retrieve(self, h_i: torch.Tensor) -> list:
        h_norm = F.normalize(h_i.unsqueeze(0), dim=-1)
        subgraph_scores = {}
        for lid in self.kg.nodes:
            embs = self.kg._get_stacked(lid)
            if embs is None or embs.shape[0] == 0:
                continue
            embs_norm    = F.normalize(embs, dim=-1)
            base_sims    = torch.mv(embs_norm, h_norm.squeeze(0))
            rarity_bonus = self.beta * self.rarity.get(lid, 0.0)
            subgraph_scores[lid] = float((base_sims + rarity_bonus).max().item())
        sorted_sgs = sorted(subgraph_scores.items(), key=lambda x: -x[1])
        selected   = [lid for lid, _ in sorted_sgs[:self.top_k]]
        results = []
        for lid in selected:
            embs = self.kg._get_stacked(lid)
            if embs is None:
                continue
            embs_norm    = F.normalize(embs, dim=-1)
            base_sims    = torch.mv(embs_norm, h_norm.squeeze(0))
            rarity_bonus = self.beta * self.rarity.get(lid, 0.0)
            boosted      = base_sims + rarity_bonus
            k1           = min(self.top_nodes_k1, embs.shape[0])
            k1_idx       = boosted.topk(k1).indices.tolist()
            k1_sim       = base_sims[k1_idx].tolist()
            seed_set     = set(k1_idx)
            if lid in self.rare_ids:
                k2     = min(self.top_nodes_k2, embs.shape[0])
                k2_idx = base_sims.topk(k2).indices.tolist()
                for idx in k2_idx:
                    if idx not in seed_set:
                        seed_set.add(idx)
                        k1_sim.append(float(base_sims[idx].item()))
                        k1_idx.append(idx)
            if self.hop >= 1:
                for idx in list(seed_set):
                    for (i, j, w) in self.kg.intra_edges.get(lid, []):
                        if i == idx and j not in seed_set:
                            seed_set.add(j)
                        elif j == idx and i not in seed_set:
                            seed_set.add(i)
                for (la, ni, lb, nj, w) in self.kg.cross_edges:
                    if la == lid and ni in seed_set:
                        ce = self.kg._get_stacked(lb)
                        if ce is not None and nj < ce.shape[0]:
                            results.append((ce[nj], w))
                    elif lb == lid and nj in seed_set:
                        ce = self.kg._get_stacked(la)
                        if ce is not None and ni < ce.shape[0]:
                            results.append((ce[ni], w))
            for idx, sim in zip(k1_idx, k1_sim):
                results.append((embs[idx], float(sim)))
        # Confusion-guided neighbours
        for lid in selected:
            if lid not in self.rare_ids:
                continue
            for edge in self._conf_by_rare.get(lid, []):
                _, _, clid, cnidx, edge_w = edge
                c_embs = self.kg._get_stacked(clid)
                if c_embs is None or cnidx >= c_embs.shape[0]:
                    continue
                results.append((c_embs[cnidx], edge_w))
        return results


# ═══════════════════════════════════════════════════════════
# DUAL KG RETRIEVER  (unchanged)
# ═══════════════════════════════════════════════════════════
class DualKGRetriever:
    def __init__(self, dual_kg: DualKnowledgeGraph, label_freqs: dict,
                 rare_ids: list):
        self.retriever_all = MinorityAwareRetriever(
            kg=dual_kg.g_all, label_freqs=label_freqs, rare_ids=rare_ids,
            top_k=KG_TOP_K, top_nodes_k1=KG_TOP_NODES,
            top_nodes_k2=KG_MIN_NODES, hop=KG_HOP, beta=RARITY_BETA)
        self.retriever_min = MinorityAwareRetriever(
            kg=dual_kg.g_min, label_freqs=label_freqs, rare_ids=rare_ids,
            top_k=KG_TOP_K, top_nodes_k1=KG_TOP_NODES,
            top_nodes_k2=KG_MIN_NODES, hop=KG_HOP, beta=RARITY_BETA * 1.5)

    def retrieve(self, h_i: torch.Tensor):
        """Returns (h_all, h_min) each as list of (Tensor(D,), float)."""
        return self.retriever_all.retrieve(h_i), self.retriever_min.retrieve(h_i)


# ═══════════════════════════════════════════════════════════
# GRAPH ATTENTION FUSION  (unchanged)
# ═══════════════════════════════════════════════════════════
class GraphAttentionFusion(nn.Module):
    def __init__(self, emb_dim=KG_FUSION_DIM, dropout=DROPOUT):
        super().__init__()
        self.proj_q  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.proj_k  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.scale   = emb_dim ** -0.5

    def forward(self, h_i, neighbours, device=None):
        if not neighbours:
            return torch.zeros_like(h_i)
        if device is None:
            device = h_i.device
        embs    = torch.stack([nb[0] for nb in neighbours]).to(device)
        weights = torch.tensor([nb[1] for nb in neighbours],
                               device=device, dtype=torch.float)
        q     = self.proj_q(h_i.unsqueeze(0))
        k     = self.proj_k(embs)
        dot   = torch.mv(k, q.squeeze(0)) * self.scale
        alpha = F.softmax(dot * weights, dim=0)
        alpha = self.dropout(alpha)
        return (alpha.unsqueeze(-1) * embs).sum(dim=0)


# ═══════════════════════════════════════════════════════════
# DYNAMIC FUSION  (unchanged)
# ═══════════════════════════════════════════════════════════
class DynamicFusion(nn.Module):
    def __init__(self, emb_dim=KG_FUSION_DIM):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(emb_dim * 3, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid())

    def forward(self, h_i, v_all, v_min, entropy):
        alpha_fixed = (ALPHA_HIGH_ENTROPY if entropy > UNCERTAINTY_THRESH
                       else ALPHA_LOW_ENTROPY)
        gate_val = self.gate(
            torch.cat([h_i, v_all, v_min], dim=-1).unsqueeze(0)).squeeze()
        alpha = 0.5 * alpha_fixed + 0.5 * gate_val.item()
        alpha = max(0.2, min(0.8, alpha))
        return alpha * v_all + (1.0 - alpha) * v_min


# ═══════════════════════════════════════════════════════════
# DUAL KG AUGMENTED MODEL  (+ ★ SCL + ★ GPU Mixup)
# ═══════════════════════════════════════════════════════════
class DualKGAugmentedModel(nn.Module):
    """
    Dual KG-RAG (G_all + G_min) + Adaptive Confusion Edges
    + ★ Supervised Contrastive Learning on fused_sent_vecs (Phase B)
    + ★ Three-Strategy GPU Manifold Mixup

    Training loss:
        (base_CRF + fused_CRF) / 2
      + AUX_CE_WEIGHT * CE
      + SCL_WEIGHT    * SupConLoss(fused_sent_vecs)   ← new
      + MIXUP_LOSS_WEIGHT * (S1 + S2 + S3)            ← new

    All SCL / Mixup branches are gated by self.training → zero
    overhead at inference.
    """
    def __init__(self, base_model, dual_kg, dual_retriever, rare_ids,
                 # ★ SCL components
                 scl_loss_fn       = None,
                 scl_class_weights = None,
                 scl_sampler       = None,
                 # ★ Mixup / Focal components
                 focal_class_weights = None,
                 confusion_pairs     = None,
                 scl_proj_dim: int   = SCL_PROJ_DIM):
        super().__init__()
        self.base      = base_model
        self.dual_kg   = dual_kg
        self.retriever = dual_retriever
        self.rare_ids  = set(rare_ids)
        self.rare_list = sorted(rare_ids)
        self.uncertainty = UncertaintyEstimator()

        sent_dim = base_model.sent_out_dim   # 256
        ctx_dim  = base_model.ctx_out_dim    # 128

        # KG-RAG fusion (unchanged)
        self.gat_all    = GraphAttentionFusion(emb_dim=sent_dim)
        self.gat_min    = GraphAttentionFusion(emb_dim=sent_dim)
        self.dyn_fusion = DynamicFusion(emb_dim=sent_dim)
        self.fusion_proj = nn.Sequential(
            nn.Linear(sent_dim, ctx_dim), nn.GELU(), nn.Dropout(DROPOUT))
        self.fusion_classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim, ctx_dim // 2), nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim // 2, NUM_LABELS))
        self.fusion_crf = CRF(num_tags=NUM_LABELS, batch_first=True)
        self.ce_loss    = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING, ignore_index=-100)

        # ★ SCL on fused embeddings
        self.fused_scl_proj = nn.Sequential(
            nn.Linear(sent_dim, sent_dim),
            nn.GELU(),
            nn.Linear(sent_dim, scl_proj_dim),
        )
        self._scl_loss_fn       = scl_loss_fn
        self._scl_class_weights = scl_class_weights
        self._scl_sampler       = scl_sampler
        self._scl_enabled       = (scl_loss_fn is not None and
                                   scl_class_weights is not None)

        # ★ Mixup components
        self.gpu_mixup  = GPUManifoldMixup(num_labels=NUM_LABELS)
        self.mixup_proj = nn.Sequential(
            nn.Linear(sent_dim, ctx_dim), nn.GELU(),
            nn.Dropout(DROPOUT), nn.Linear(ctx_dim, NUM_LABELS))
        self.soft_ce = SoftLabelCrossEntropy(
            temperature=MIXUP_SOFT_TEMP,
            class_weights=focal_class_weights)

        # Rare label tensor (buffer for .to(device) support)
        self.register_buffer("_rare_t",
                             torch.tensor(self.rare_list, dtype=torch.long))
        # Confused majority ids for S3 inter-class mixup
        conf_maj_ids = list({cid for cids in (confusion_pairs or {}).values()
                             for cid in cids})
        if not conf_maj_ids:
            conf_maj_ids = [i for i in range(NUM_LABELS)
                            if i not in self.rare_ids]
        self.register_buffer("_conf_maj_t",
                             torch.tensor(conf_maj_ids, dtype=torch.long))

    # ── SCL helper on fused embeddings ───────────────────
    def _get_fused_scl_embs(self, fused_sent, labels):
        """Project fused_sent → L2-normalised contrastive embeddings."""
        B, T, D = fused_sent.shape
        flat_vecs   = fused_sent.reshape(B * T, D)
        flat_labels = labels.reshape(B * T)
        valid       = flat_labels != -100
        flat_vecs   = flat_vecs[valid]
        flat_labels = flat_labels[valid]
        if flat_vecs.shape[0] == 0:
            return None, None
        proj = F.normalize(self.fused_scl_proj(flat_vecs), dim=-1)
        return proj, flat_labels

    # ── KG fusion batch (unchanged) ──────────────────────
    def _fuse_batch(self, sent_vecs, emissions, lengths, device):
        B, T, sent_dim = sent_vecs.shape
        fused      = sent_vecs.clone()
        entropy_map    = self.uncertainty.entropy(emissions)
        uncertain_mask = entropy_map > UNCERTAINTY_THRESH
        top_labels     = self.uncertainty.top_label(emissions)
        for b in range(B):
            n = int(lengths[b].item())
            for t in range(n):
                uncertain = bool(uncertain_mask[b, t].item())
                pred_lbl  = int(top_labels[b, t].item())
                is_rare   = pred_lbl in self.rare_ids
                if not (uncertain or (RARE_ALWAYS_KG and is_rare)):
                    continue
                h_i_cpu = sent_vecs[b, t].detach().cpu()
                h_all, h_min = self.retriever.retrieve(h_i_cpu)
                if not h_all and not h_min:
                    continue
                h_i_gpu = sent_vecs[b, t]
                ent_val = float(entropy_map[b, t].item())
                v_all = (self.gat_all(h_i_gpu, h_all, device=device)
                         if h_all else torch.zeros(sent_dim, device=device))
                v_min = (self.gat_min(h_i_gpu, h_min, device=device)
                         if h_min else torch.zeros(sent_dim, device=device))
                v_i = self.dyn_fusion(h_i_gpu, v_all, v_min, ent_val)
                fused[b, t] = h_i_gpu + v_i
        return fused

    # ── ★ Mixup auxiliary losses ──────────────────────────
    def _compute_mixup_losses(self, sent_vecs, labels,
                               lengths, device) -> torch.Tensor:
        """
        Compute combined Mixup loss from strategies S1 + S2 + S3.
        All tensor operations are on GPU; CPU is only used for the
        DualKGRetriever call in S2 (capped at KG_MIXUP_MAX_PER_BATCH).
        """
        total = torch.tensor(0.0, device=device)
        B, T, D = sent_vecs.shape

        # Flatten valid sentences
        flat_embs, flat_labs = [], []
        for b in range(B):
            n = int(lengths[b].item())
            flat_embs.append(sent_vecs[b, :n])
            flat_labs.append(labels[b, :n])
        if not flat_embs:
            return total
        flat_embs = torch.cat(flat_embs, dim=0)
        flat_labs = torch.cat(flat_labs, dim=0)
        valid_m   = flat_labs >= 0
        flat_embs = flat_embs[valid_m]
        flat_labs = flat_labs[valid_m]
        if flat_embs.shape[0] < MIXUP_MIN_RARE_IN_BATCH:
            return total

        # ── S1: Intra-Rare Mixup ──────────────────────────
        if INTRA_RARE_MIXUP_ENABLED:
            mx_embs, soft = self.gpu_mixup.intra_rare_mixup(
                flat_embs, flat_labs, self._rare_t)
            if mx_embs is not None:
                logits  = self.mixup_proj(mx_embs)
                s1_loss = self.soft_ce(logits, soft.to(device))
                total   = total + INTRA_RARE_MIXUP_WEIGHT * s1_loss

        # ── S2: KG-Guided Mixup (uses DualKGRetriever) ────
        if KG_MIXUP_ENABLED:
            s2_embs, s2_soft = [], []
            n_s2 = 0
            for b in range(B):
                n = int(lengths[b].item())
                for t in range(n):
                    if n_s2 >= KG_MIXUP_MAX_PER_BATCH:
                        break
                    l_bt = int(labels[b, t].item())
                    if l_bt < 0 or l_bt not in self.rare_ids:
                        continue
                    h_i_cpu = sent_vecs[b, t].detach().cpu()
                    h_all, h_min = self.retriever.retrieve(h_i_cpu)
                    all_nbs = h_all + h_min
                    h_i_gpu = sent_vecs[b, t]
                    mixed, soft_i = self.gpu_mixup.kg_guided_mixup_from_list(
                        h_i_gpu, l_bt, all_nbs)
                    if mixed is not None:
                        s2_embs.append(mixed)
                        s2_soft.append(soft_i.to(device))
                        n_s2 += 1
                if n_s2 >= KG_MIXUP_MAX_PER_BATCH:
                    break
            if s2_embs:
                s2_embs_t = torch.stack(s2_embs)
                s2_soft_t = torch.stack(s2_soft)
                s2_logits = self.mixup_proj(s2_embs_t)
                s2_loss   = self.soft_ce(s2_logits, s2_soft_t)
                total     = total + KG_MIXUP_WEIGHT * s2_loss

        # ── S3: Rare ↔ Hard-Majority Inter-class Mixup ───
        if INTER_MIXUP_ENABLED:
            mx_embs, soft = self.gpu_mixup.inter_class_mixup(
                flat_embs, flat_labs, self._rare_t, self._conf_maj_t)
            if mx_embs is not None:
                logits  = self.mixup_proj(mx_embs)
                s3_loss = self.soft_ce(logits, soft.to(device))
                total   = total + INTER_MIXUP_WEIGHT * s3_loss

        return total

    # ── Forward ───────────────────────────────────────────
    def forward(self, input_ids, attention_mask, token_type_ids,
                labels=None, lengths=None):
        device = input_ids.device

        sent_vecs, _, base_emissions = self.base.get_emissions(
            input_ids, attention_mask, token_type_ids, lengths=lengths)

        fused_sent = self._fuse_batch(
            sent_vecs, base_emissions, lengths, device)

        fused_drop = self.base.dropout(fused_sent)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                fused_drop, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out, _ = self.base.ctx_bilstm(packed)
            fused_ctx, _  = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True)
        else:
            fused_ctx, _ = self.base.ctx_bilstm(fused_drop)
        fused_ctx = self.base.dropout(fused_ctx)

        fused_emissions = self.fusion_classifier(fused_ctx)
        fused_emissions = torch.nan_to_num(
            fused_emissions, nan=0.0, posinf=1e4, neginf=-1e4)

        if lengths is not None:
            B, T, _ = fused_emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(fused_emissions.shape[:2],
                              dtype=torch.bool, device=device)

        combined = (base_emissions + fused_emissions) / 2.0

        if labels is not None:
            safe = labels.clone(); safe[safe == -100] = 0
            base_crf_loss  = -self.base.crf(
                base_emissions, safe, mask=mask, reduction="mean")
            fused_crf_loss = -self.fusion_crf(
                fused_emissions, safe, mask=mask, reduction="mean")
            B2, T2, C = combined.shape
            ce_loss = self.ce_loss(
                combined.reshape(B2*T2, C), labels.reshape(B2*T2))

            total_loss = ((base_crf_loss + fused_crf_loss) / 2.0
                          + AUX_CE_WEIGHT * ce_loss)

            # ★ SCL on fused embeddings (training only)
            if self.training and self._scl_enabled:
                proj_f, flat_lf = self._get_fused_scl_embs(fused_sent, labels)
                if proj_f is not None and proj_f.shape[0] >= 2:
                    if self._scl_sampler is not None:
                        proj_f, flat_lf = self._scl_sampler.sample(proj_f, flat_lf)
                    cw = self._scl_class_weights.to(device)
                    scl_loss = self._scl_loss_fn(
                        proj_f, flat_lf, cw, rare_ids=self.rare_list)
                    total_loss = total_loss + SCL_WEIGHT * scl_loss

            # ★ Mixup auxiliary losses (training only)
            if self.training:
                mixup_loss = self._compute_mixup_losses(
                    sent_vecs, labels, lengths, device)
                total_loss = total_loss + MIXUP_LOSS_WEIGHT * mixup_loss

            return total_loss, combined
        else:
            return self.fusion_crf.decode(fused_emissions, mask=mask), fused_emissions


# ═══════════════════════════════════════════════════════════
# METRICS
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)
    macro_prec    = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec    = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_prec = precision_score(all_trues, all_preds, average="weighted", zero_division=0)
    macro_rec    = recall_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_rec    = recall_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_rec = recall_score(all_trues, all_preds, average="weighted", zero_division=0)
    acc = accuracy_score(all_trues, all_preds)
    per_class_f1   = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                              average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds,
                                     labels=list(range(NUM_LABELS)),
                                     average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds,
                                  labels=list(range(NUM_LABELS)),
                                  average=None, zero_division=0)
    per_class_metrics = {
        id2label[i]: {"f1": float(per_class_f1[i]),
                      "precision": float(per_class_prec[i]),
                      "recall":    float(per_class_rec[i])}
        for i in range(NUM_LABELS)
    }
    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rare_f1   = f1_score(all_trues, all_preds, labels=present_rare,
                             average="macro", zero_division=0)
        rare_prec = precision_score(all_trues, all_preds, labels=present_rare,
                                    average="macro", zero_division=0)
        rare_rec  = recall_score(all_trues, all_preds, labels=present_rare,
                                 average="macro", zero_division=0)
    else:
        rare_f1 = rare_prec = rare_rec = 0.0
    str_t = [id2label[x] for x in all_trues]
    str_p = [id2label[x] for x in all_preds]
    cls_report = classification_report(str_t, str_p, labels=LABELS,
                                        digits=4, zero_division=0)
    cm = confusion_matrix(str_t, str_p, labels=LABELS)
    return dict(
        macro_f1=macro_f1, micro_f1=micro_f1, weighted_f1=weighted_f1,
        macro_precision=macro_prec, micro_precision=micro_prec,
        weighted_precision=weighted_prec,
        macro_recall=macro_rec, micro_recall=micro_rec,
        weighted_recall=weighted_rec,
        rare_f1=rare_f1, rare_precision=rare_prec, rare_recall=rare_rec,
        per_class_metrics=per_class_metrics,
        accuracy=acc, cls_report=cls_report, cm=cm,
        all_preds=all_preds, all_trues=all_trues,
    )


def count_parameters(model):
    tr = sum(p.numel() for p in model.parameters() if     p.requires_grad)
    fr = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"\n  Trainable: {tr:,} | Frozen: {fr:,}")
    return tr, fr


# ═══════════════════════════════════════════════════════════
# BASE TRAINER  (Phase A  +  ★ FocalLoss  +  ★ WeightedSampler)
# ═══════════════════════════════════════════════════════════
class BaseTrainer:
    """
    Phase A trainer — trains InLegalBERT_BiLSTM_MHA_CRF.
    New vs original:
      • Accepts focal_loss for auxiliary focal CE
      • Uses WeightedRandomSampler (rare over-sampling)
      • SCL is computed internally in model.forward() (via stored components)
    """
    def __init__(self, model, focal_loss: FocalLoss = None, device=DEVICE):
        self.model      = model.to(device)
        self.focal_loss = focal_loss
        self.device     = device

    def build_optimizer(self):
        pg = []
        pg.append({"params": list(self.model.bert.pooler.parameters()),
                   "lr": BERT_LR, "weight_decay": WEIGHT_DECAY})
        enc = self.model.bert.encoder.layer
        n   = len(enc)
        for i in range(n - 1, BERT_FREEZE_LAYERS - 1, -1):
            depth  = (n - 1) - i
            lr_i   = BERT_LR * (BERT_LR_DECAY ** depth)
            params = [p for p in enc[i].parameters() if p.requires_grad]
            if params:
                pg.append({"params": params, "lr": lr_i,
                           "weight_decay": WEIGHT_DECAY})
        head_modules = [
            self.model.sent_bilstm, self.model.mha_pooling,
            self.model.sent_layer_norm, self.model.ctx_bilstm,
            self.model.classifier, self.model.crf,
            self.model.scl_proj,   # ★ SCL projection head
        ]
        head_params = [p for m in head_modules for p in m.parameters()]
        pg.append({"params": head_params, "lr": HEAD_LR,
                   "weight_decay": WEIGHT_DECAY})
        return torch.optim.AdamW(pg)

    def compute_val_loss(self, dataset):
        # model.eval() → self.training=False → SCL skipped automatically
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        total, n = 0.0, 0
        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids = ids.to(self.device); attn  = attn.to(self.device)
                ttype = ttype.to(self.device); labels = labels.to(self.device)
                lengths = lengths.to(self.device)
                loss, _ = self.model(ids, attn, ttype,
                                     labels=labels, lengths=lengths)
                if not torch.isnan(loss):
                    total += loss.item(); n += 1
        return total / max(1, n)

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples   = len(dataset)
        infer_start = time.time() if measure_inference_time else None
        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids     = ids.to(self.device); attn    = attn.to(self.device)
                ttype   = ttype.to(self.device); lengths = lengths.to(self.device)
                decoded, _ = self.model(ids, attn, ttype,
                                        labels=None, lengths=lengths)
                for i, seq in enumerate(decoded):
                    tl = int(lengths[i].item())
                    all_preds.extend(seq)
                    all_trues.extend(labels[i, :tl].tolist())
        if measure_inference_time:
            ti  = time.time() - infer_start
            ns  = len(all_trues)
            with open(os.path.join(OUT_DIR,
                                   f"inference_time_{split_name}.json"), "w") as f:
                json.dump({"total_inference_time_s": ti,
                           "latency_per_document_ms": ti/max(1,n_samples)*1000,
                           "throughput_sentences_per_s": ns/max(1e-9,ti)}, f, indent=2)
        return compute_all_metrics(all_trues, all_preds, rare_ids, split_name)

    def train(self, train_docs, train_dataset, dev_dataset, rare_ids,
              tokenizer, num_epochs=NUM_EPOCHS_BASE):
        # ★ Weighted sampler for rare over-sampling
        sampler      = build_weighted_sampler(train_docs, rare_ids)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                  sampler=sampler, collate_fn=collate_rrc)
        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(
            optimizer, warmup_steps, total_steps)
        early_stopper = EarlyStopping()
        history       = []
        best_f1, best_state = -1.0, None
        total_start   = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            running_loss, n_steps = 0.0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for step, (ids, attn, ttype, labels, lengths) in enumerate(train_loader):
                ids     = ids.to(self.device);   attn    = attn.to(self.device)
                ttype   = ttype.to(self.device); labels  = labels.to(self.device)
                lengths = lengths.to(self.device)

                # model.forward() includes CRF + CE + SCL internally
                loss, emissions = self.model(ids, attn, ttype,
                                             labels=labels, lengths=lengths)

                # ★ Additional focal CE for auxiliary class-weighted supervision
                if self.focal_loss is not None:
                    B2, T2, C = emissions.shape
                    fl = self.focal_loss(
                        emissions.reshape(B2*T2, C).detach(),
                        labels.reshape(B2*T2))
                    loss = loss + AUX_CE_WEIGHT * fl

                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad(); continue

                (loss / GRADIENT_ACCUMULATION_STEPS).backward()
                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(
                        self.model.parameters(), GRAD_CLIP)
                    optimizer.step(); scheduler.step(); optimizer.zero_grad()
                running_loss += loss.item(); n_steps += 1

            if n_steps % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

            epoch_time  = time.time() - epoch_start
            avg_loss    = running_loss / max(1, n_steps)
            val_loss    = self.compute_val_loss(dev_dataset)
            val_metrics = self.evaluate(dev_dataset, rare_ids)

            print(
                f"[Base+SCL] Ep {epoch:03d}/{num_epochs} | "
                f"loss:{avg_loss:.4f} val_loss:{val_loss:.4f} | "
                f"mac_F1:{val_metrics['macro_f1']:.4f} "
                f"rare_F1:{val_metrics['rare_f1']:.4f} | "
                f"t:{epoch_time:.1f}s "
                f"ES:{early_stopper.counter}/{early_stopper.patience}"
            )
            history.append({
                "epoch": epoch, "phase": "base_scl",
                "train_loss": avg_loss, "val_loss": val_loss,
                "val_macro_f1": val_metrics["macro_f1"],
                "val_rare_f1":  val_metrics["rare_f1"],
                "val_accuracy": val_metrics["accuracy"],
                "epoch_train_time_s": epoch_time,
            })
            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f}")
            if early_stopper.step(val_metrics["macro_f1"]):
                print(f"\n⏹  Base early stopping at epoch {epoch}.\n"); break

        total_time = time.time() - total_start
        pd.DataFrame(history).to_csv(
            os.path.join(OUT_DIR, "base_history.csv"), index=False)
        if best_state:
            self.model.load_state_dict(best_state)
            print(f"\n✔ Best base model restored (val_macro_f1={best_f1:.4f})")
            self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
            tokenizer.save_pretrained(BEST_MODEL_DIR)
            torch.save(best_state,
                       os.path.join(BEST_MODEL_DIR, "base_scl_model.bin"))
        return pd.DataFrame(history), total_time


# ═══════════════════════════════════════════════════════════
# DUAL KG BUILDER  (unchanged)
# ═══════════════════════════════════════════════════════════
@torch.no_grad()
def build_dual_knowledge_graph(
        base_model, train_docs, tokenizer, rare_ids,
        confusion_pairs=None, confusion_weights=None,
        device=DEVICE) -> DualKnowledgeGraph:
    print("\n🔨 Building Dual Knowledge Graph (G_all + G_min) ...")
    base_model.eval(); base_model.to(device)
    dual_kg       = DualKnowledgeGraph(rare_ids=rare_ids,
                                        emb_dim=base_model.sent_out_dim)
    dummy_dataset = RRCDataset(train_docs, tokenizer)
    loader        = DataLoader(dummy_dataset, batch_size=1,
                               shuffle=False, collate_fn=collate_rrc)
    for doc_idx, (ids, attn, ttype, labels, lengths) in enumerate(loader):
        ids   = ids.to(device); attn = attn.to(device); ttype = ttype.to(device)
        lengths = lengths.to(device)
        sent_vecs = base_model.encode_sentences(ids, attn, ttype).squeeze(0)
        n         = int(lengths[0].item())
        dual_kg.add_nodes(sent_vecs[:n].cpu(), labels[0, :n].tolist())
        if (doc_idx + 1) % 50 == 0:
            print(f"  Processed {doc_idx+1}/{len(loader)} docs")
    dual_kg.build_edges()
    if confusion_pairs is not None and confusion_weights is not None:
        dual_kg.build_confusion_edges(confusion_pairs, confusion_weights)
    else:
        print("  ⚠ Skipping confusion edges (no confusion data provided).")
    dual_kg.save(OUT_DIR)
    n_all = sum(len(v) for v in dual_kg.g_all.nodes.values())
    n_min = sum(len(v) for v in dual_kg.g_min.nodes.values())
    print(f"  G_all: {n_all} nodes, {len(dual_kg.g_all.conf_cx_edges)} confusion edges")
    print(f"  G_min: {n_min} nodes, {len(dual_kg.g_min.conf_cx_edges)} confusion edges")
    return dual_kg


# ═══════════════════════════════════════════════════════════
# KG TRAINER  (Phase B  +  ★ SCL  +  ★ Mixup  +  ★ HardReplay)
# ═══════════════════════════════════════════════════════════
class KGTrainer:
    """
    Phase B trainer — fine-tunes DualKGAugmentedModel.
    New vs original:
      • Uses WeightedRandomSampler (rare over-sampling)
      • HardExampleBuffer for hard-example replay
      • SCL + Mixup run internally in model.forward() (training only)
    """
    def __init__(self, kg_model: DualKGAugmentedModel, device=DEVICE):
        self.model  = kg_model.to(device)
        self.device = device

    def build_optimizer(self):
        new_params = (
            list(self.model.gat_all.parameters()) +
            list(self.model.gat_min.parameters()) +
            list(self.model.dyn_fusion.parameters()) +
            list(self.model.fusion_proj.parameters()) +
            list(self.model.fusion_classifier.parameters()) +
            list(self.model.fusion_crf.parameters()) +
            list(self.model.fused_scl_proj.parameters()) +  # ★ SCL
            list(self.model.mixup_proj.parameters()) +       # ★ Mixup
            list(self.model.gpu_mixup.parameters())          # ★ Mixup
        )
        base_trainable = [p for p in self.model.base.parameters()
                          if p.requires_grad]
        return torch.optim.AdamW([
            {"params": new_params,     "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY},
            {"params": base_trainable, "lr": BERT_LR, "weight_decay": WEIGHT_DECAY},
        ])

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples   = len(dataset)
        infer_start = time.time() if measure_inference_time else None
        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids     = ids.to(self.device); attn    = attn.to(self.device)
                ttype   = ttype.to(self.device); lengths = lengths.to(self.device)
                decoded, _ = self.model(ids, attn, ttype,
                                        labels=None, lengths=lengths)
                for i, sp in enumerate(decoded):
                    tl = int(lengths[i].item())
                    all_preds.extend(sp)
                    all_trues.extend(labels[i, :tl].tolist())
        if measure_inference_time:
            ti  = time.time() - infer_start
            ns  = len(all_trues)
            with open(os.path.join(OUT_DIR,
                                   f"kg_inference_{split_name}.json"), "w") as f:
                json.dump({"total_inference_time_s": ti,
                           "latency_per_document_ms": ti/max(1,n_samples)*1000,
                           "throughput_sentences_per_s": ns/max(1e-9,ti)}, f, indent=2)
        return compute_all_metrics(all_trues, all_preds, rare_ids, split_name)

    def train(self, train_docs, train_dataset, dev_dataset,
              rare_ids, num_epochs=NUM_EPOCHS_KG):
        # ★ Rare over-sampling
        sampler      = build_weighted_sampler(train_docs, rare_ids)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                  sampler=sampler, collate_fn=collate_rrc)
        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(
            optimizer, warmup_steps, total_steps)
        early_stopper = EarlyStopping(patience=5)
        # ★ Hard example replay
        hard_buffer   = HardExampleBuffer(max_size=HARD_BUFFER_SIZE)
        rare_set      = set(rare_ids)
        history       = []
        best_f1, best_state = -1.0, None
        total_start   = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            running_loss, n_steps = 0.0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for step, (ids, attn, ttype, labels, lengths) in enumerate(train_loader):
                ids     = ids.to(self.device);   attn    = attn.to(self.device)
                ttype   = ttype.to(self.device); labels  = labels.to(self.device)
                lengths = lengths.to(self.device)

                # forward includes CRF + CE + SCL + Mixup
                loss, _ = self.model(ids, attn, ttype,
                                     labels=labels, lengths=lengths)
                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad(); continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()
                running_loss += loss.item(); n_steps += 1

                # ★ Buffer hard examples with rare classes
                lv = loss.item()
                if lv > HARD_REPLAY_LOSS_THRESH:
                    flat_l = labels.cpu().flatten()
                    if any(int(x) in rare_set for x in flat_l if x.item() >= 0):
                        hard_buffer.update(lv, (ids.cpu(), attn.cpu(),
                                               ttype.cpu(), labels.cpu(),
                                               lengths.cpu()))

                # ★ Hard example replay
                if (step + 1) % HARD_REPLAY_FREQ == 0 and len(hard_buffer) >= 10:
                    replay = hard_buffer.sample()
                    if replay is not None:
                        r_ids, r_attn, r_tt, r_lab, r_len = [
                            x.to(self.device) for x in replay]
                        r_loss, _ = self.model(r_ids, r_attn, r_tt,
                                               labels=r_lab, lengths=r_len)
                        if not torch.isnan(r_loss) and not torch.isinf(r_loss):
                            (r_loss * HARD_REPLAY_WEIGHT).backward()
                            torch.nn.utils.clip_grad_norm_(
                                self.model.parameters(), GRAD_CLIP)
                            optimizer.step(); scheduler.step(); optimizer.zero_grad()

            epoch_time  = time.time() - epoch_start
            avg_loss    = running_loss / max(1, n_steps)
            val_metrics = self.evaluate(dev_dataset, rare_ids)

            print(
                f"[DualKG+SCL+Mixup] Ep {epoch:02d}/{num_epochs} | "
                f"loss:{avg_loss:.4f} | "
                f"mac_F1:{val_metrics['macro_f1']:.4f} "
                f"rare_F1:{val_metrics['rare_f1']:.4f} | "
                f"t:{epoch_time:.1f}s buf:{len(hard_buffer)} "
                f"ES:{early_stopper.counter}/{early_stopper.patience}"
            )
            history.append({
                "epoch": epoch, "phase": "dual_kg_scl_mixup",
                "train_loss": avg_loss,
                "val_macro_f1": val_metrics["macro_f1"],
                "val_rare_f1":  val_metrics["rare_f1"],
                "val_accuracy": val_metrics["accuracy"],
                "epoch_train_time_s": epoch_time,
            })
            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best DualKG+SCL+Mixup val_macro_f1={best_f1:.4f}")
            if early_stopper.step(val_metrics["macro_f1"]):
                print(f"\n⏹  DualKG+SCL+Mixup early stopping at ep {epoch}.\n")
                break

        total_time = time.time() - total_start
        pd.DataFrame(history).to_csv(
            os.path.join(OUT_DIR, "dual_kg_scl_mixup_history.csv"), index=False)
        if best_state:
            self.model.load_state_dict(best_state)
            torch.save(best_state,
                       os.path.join(BEST_MODEL_DIR, "dual_kg_scl_mixup_model.bin"))
            print(f"\n✔ Best model saved (val_macro_f1={best_f1:.4f})")
        return pd.DataFrame(history), total_time


# ═══════════════════════════════════════════════════════════
# VISUALISATION
# ═══════════════════════════════════════════════════════════
def save_confusion_matrix(cm, split_name, rare_labels=None):
    fig, ax = plt.subplots(figsize=(14, 11))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=LABELS, yticklabels=LABELS,
                cmap="Blues", ax=ax)
    if rare_labels:
        for tick in ax.get_xticklabels() + ax.get_yticklabels():
            if tick.get_text() in rare_labels:
                tick.set_color("red")
    ax.set_title(f"{split_name.capitalize()} Confusion Matrix "
                 f"(DualKG-RAG + AdaptiveEdge + SCL + Mixup)")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png")
    plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
    f1s    = [per_class_metrics[l]["f1"] for l in LABELS]
    colors = ["tomato" if (rare_labels and l in rare_labels)
              else "steelblue" for l in LABELS]
    fig, ax = plt.subplots(figsize=(9, 6))
    bars = ax.barh(LABELS, f1s, color=colors, edgecolor="white")
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_xlim(0, 1.12); ax.set_xlabel("F1 Score")
    ax.set_title(f"{split_name.capitalize()} Per-Class F1 "
                 f"(DualKG+SCL+Mixup)")
    ax.grid(True, alpha=0.3, axis="x")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png")
    plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


def plot_combined_history(base_df, kg_df):
    all_df = pd.concat([base_df, kg_df], ignore_index=True)
    all_df["global_epoch"] = range(1, len(all_df) + 1)
    boundary = len(base_df)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    ax.plot(all_df["global_epoch"], all_df["train_loss"],
            marker="o", markersize=3, label="Train Loss")
    ax.axvline(boundary, color="red", linestyle="--", label="Phase B start")
    ax.set_title("Training Loss"); ax.legend(); ax.grid(True, alpha=0.3)
    ax = axes[1]
    ax.plot(all_df["global_epoch"], all_df["val_macro_f1"],
            label="Val Macro-F1", marker="o", markersize=3)
    ax.plot(all_df["global_epoch"], all_df["val_rare_f1"],
            label="Val Rare-F1", marker="s", markersize=3)
    ax.axvline(boundary, color="red", linestyle="--", label="Phase B start")
    ax.set_title("Val F1 (Base+SCL → DualKG+SCL+Mixup)")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path = os.path.join(OUT_DIR, "combined_training_curves.png")
    plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


def print_metrics_table(dev_metrics, test_metrics,
                        base_time=None, kg_time=None, total_trainable=None):
    rows = [
        ("Accuracy",           "accuracy"),
        ("Macro-F1",           "macro_f1"),
        ("Micro-F1",           "micro_f1"),
        ("Weighted-F1",        "weighted_f1"),
        ("Rare / Minority F1", "rare_f1"),
        ("Macro-Precision",    "macro_precision"),
        ("Macro-Recall",       "macro_recall"),
    ]
    print("\n" + "=" * 76)
    print("FINAL RESULTS — DualKG-RAG + Adaptive Confusion Edges + SCL + Mixup")
    print("=" * 76)
    if total_trainable: print(f"  Trainable Parameters : {total_trainable:,}")
    if base_time:        print(f"  Phase A time         : {base_time/60:.1f} min")
    if kg_time:          print(f"  Phase B time         : {kg_time/60:.1f} min")
    print("-" * 76)
    print(f"  {'Metric':<28} {'Dev':>12} {'Test':>12}")
    print("-" * 76)
    for label, key in rows:
        print(f"  {label:<28} {dev_metrics[key]:>12.4f} {test_metrics[key]:>12.4f}")
    print("=" * 76)
    print("\n  PER-CLASS F1 / PRECISION / RECALL")
    print("  " + "-" * 66)
    print(f"  {'Label':<22} {'F1-Dev':>9} {'F1-Test':>9} "
          f"{'Prec-Test':>11} {'Rec-Test':>10}")
    print("  " + "-" * 66)
    for lbl in LABELS:
        dv = dev_metrics["per_class_metrics"][lbl]
        ts = test_metrics["per_class_metrics"][lbl]
        print(f"  {lbl:<22} {dv['f1']:>9.4f} {ts['f1']:>9.4f} "
              f"{ts['precision']:>11.4f} {ts['recall']:>10.4f}")
    print("  " + "-" * 66)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device : {DEVICE}")
    print("Architecture : InLegalBERT + BiLSTM + MHA + CRF")
    print("               → Dual KG-RAG (G_all + G_min) + Adaptive Confusion Edges")
    print("               + Supervised Contrastive Learning (SCL)")
    print("               + GPU Manifold Mixup (S1 IntraRare | S2 KG-Guided | S3 Inter-class)")
    print("               + Focal Loss  +  Rare Oversampling  +  Hard Replay\n")

    # ── Data ──────────────────────────────────────────────
    print("Loading JSONL files ...")
    train_docs = extract_docs(load_jsonl(TRAIN_PATH))
    dev_docs   = extract_docs(load_jsonl(DEV_PATH))
    test_docs  = extract_docs(load_jsonl(TEST_PATH))
    print(f"  Train:{len(train_docs)} | Dev:{len(dev_docs)} | Test:{len(test_docs)}")

    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)
    pd.DataFrame([{"label": l, "frequency": label_freqs[l],
                   "is_rare": l in rare_labels}
                  for l in LABELS]
                 ).to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)

    # ★ Class weights (two flavours)
    scl_class_weights   = compute_scl_class_weights(label_freqs, rare_ids)
    focal_class_weights = compute_focal_class_weights(label_freqs)

    # ★ SCL components
    scl_loss_fn = SupConLoss(temperature=SCL_TEMPERATURE,
                              minority_boost=SCL_MINORITY_BOOST)
    scl_sampler = BalancedContrastiveSampler(samples_per_class=SCL_SAMPLES_PER_CLASS,
                                             rare_ids=rare_ids)

    # ★ Focal Loss
    focal_loss = FocalLoss(focal_class_weights, rare_ids).to(DEVICE)
    print(f"\n  Focal class weights (top-5 rare): "
          f"{[(id2label[r], round(float(focal_class_weights[r]),2)) for r in rare_ids[:5]]}")

    print("Loading tokenizer ...")
    tokenizer     = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)
    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    # ════════════════════════════════════════════════════
    # PHASE A: Base Model Training  (+SCL +FocalLoss +Oversampling)
    # ════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A: Base Model Training  (CRF + CE + SCL + Focal)")
    print(f"  SCL weight={SCL_WEIGHT}  temperature={SCL_TEMPERATURE}  "
          f"minority_boost={SCL_MINORITY_BOOST}x")
    print(f"  Oversample rare ratio = {OVERSAMPLE_RARE_RATIO}x")
    print("=" * 60)

    base_model = InLegalBERT_BiLSTM_MHA_CRF(
        scl_loss_fn       = scl_loss_fn,
        scl_class_weights = scl_class_weights,
        scl_sampler       = scl_sampler,
        rare_ids          = rare_ids,
    )
    base_trainer = BaseTrainer(base_model, focal_loss=focal_loss, device=DEVICE)
    base_hist_df, base_time = base_trainer.train(
        train_docs, train_dataset, dev_dataset, rare_ids,
        tokenizer=tokenizer, num_epochs=NUM_EPOCHS_BASE)

    # ════════════════════════════════════════════════════
    # PHASE A→B: Confusion Analysis + Adaptive Weights
    # ════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A→B: Confusion Analysis + Adaptive Weight Derivation")
    print("=" * 60)
    confusion_pairs, raw_cm = compute_confusion_pairs_and_matrix(
        base_model, train_dataset, rare_ids, device=DEVICE, top_k=CONF_TOP_K)
    confusion_weights = compute_adaptive_confusion_weights(
        raw_cm, rare_ids, base_alpha=CONF_BASE_ALPHA)
    save_adaptive_weight_heatmap(confusion_weights, rare_ids, out_dir=OUT_DIR)
    np.save(os.path.join(OUT_DIR, "confusion_weights.npy"),
            confusion_weights.numpy())
    np.save(os.path.join(OUT_DIR, "raw_confusion_matrix.npy"), raw_cm)
    conf_pairs_serial = {str(k): [int(v) for v in vl]
                         for k, vl in confusion_pairs.items()}
    with open(os.path.join(OUT_DIR, "confusion_pairs.json"), "w") as f:
        json.dump(conf_pairs_serial, f, indent=2)

    # ════════════════════════════════════════════════════
    # PHASE A→B: Build Dual KG  (RST + Confusion Edges)
    # ════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A→B: Building Dual Knowledge Graph + Confusion Edges")
    print("=" * 60)
    kg_global_path   = os.path.join(OUT_DIR, "kg_global.json")
    kg_minority_path = os.path.join(OUT_DIR, "kg_minority.json")
    if os.path.exists(kg_global_path) and os.path.exists(kg_minority_path):
        print("  Found cached KG — checking confusion edges ...")
        dual_kg = DualKnowledgeGraph.load(
            OUT_DIR, rare_ids=rare_ids, emb_dim=base_model.sent_out_dim)
        if len(dual_kg.g_all.conf_cx_edges) == 0 and len(confusion_pairs) > 0:
            print("  ⚠ No confusion edges in cache — rebuilding ...")
            dual_kg = build_dual_knowledge_graph(
                base_model, train_docs, tokenizer, rare_ids,
                confusion_pairs=confusion_pairs,
                confusion_weights=confusion_weights, device=DEVICE)
        else:
            print(f"  Refreshing confusion edges with current weights ...")
            dual_kg.build_confusion_edges(confusion_pairs, confusion_weights)
            dual_kg.save(OUT_DIR)
    else:
        dual_kg = build_dual_knowledge_graph(
            base_model, train_docs, tokenizer, rare_ids,
            confusion_pairs=confusion_pairs,
            confusion_weights=confusion_weights, device=DEVICE)

    # ════════════════════════════════════════════════════
    # PHASE B: DualKG + SCL + Mixup Fine-Tuning
    # ════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE B: DualKG + Adaptive Edges + SCL + GPU Mixup Fine-Tuning")
    print(f"  Mixup α={MIXUP_ALPHA}  KG-Mixup α={KG_MIXUP_ALPHA}  "
          f"Inter-class α={INTER_MIXUP_ALPHA}")
    print(f"  Loss weights: S1={INTRA_RARE_MIXUP_WEIGHT} "
          f"S2={KG_MIXUP_WEIGHT} S3={INTER_MIXUP_WEIGHT} "
          f"SCL={SCL_WEIGHT}")
    print(f"  G_all confusion edges: {len(dual_kg.g_all.conf_cx_edges)}")
    print(f"  G_min confusion edges: {len(dual_kg.g_min.conf_cx_edges)}")
    print("=" * 60)

    dual_retriever = DualKGRetriever(
        dual_kg=dual_kg, label_freqs=label_freqs, rare_ids=rare_ids)

    kg_model = DualKGAugmentedModel(
        base_model          = base_model,
        dual_kg             = dual_kg,
        dual_retriever      = dual_retriever,
        rare_ids            = rare_ids,
        scl_loss_fn         = scl_loss_fn,
        scl_class_weights   = scl_class_weights,
        scl_sampler         = scl_sampler,
        focal_class_weights = focal_class_weights.to(DEVICE),
        confusion_pairs     = confusion_pairs,
    )
    total_trainable, _ = count_parameters(kg_model)

    kg_trainer = KGTrainer(kg_model, device=DEVICE)
    kg_hist_df, kg_time = kg_trainer.train(
        train_docs, train_dataset, dev_dataset,
        rare_ids=rare_ids, num_epochs=NUM_EPOCHS_KG)

    plot_combined_history(base_hist_df, kg_hist_df)

    # ════════════════════════════════════════════════════
    # EVALUATION
    # ════════════════════════════════════════════════════
    print("\nEvaluating on Dev set ...")
    dev_metrics = kg_trainer.evaluate(dev_dataset, rare_ids,
                                       split_name="dev",
                                       measure_inference_time=True)
    print(f"  Dev  Accuracy:{dev_metrics['accuracy']:.4f} "
          f"Macro-F1:{dev_metrics['macro_f1']:.4f} "
          f"Rare-F1:{dev_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "dev_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT+BiLSTM+MHA+CRF+DualKG-RAG+AdaptiveEdge+SCL+Mixup\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n")
        f.write(f"SCL_WEIGHT={SCL_WEIGHT}  MIXUP_ALPHA={MIXUP_ALPHA}  "
                f"CONF_BASE_ALPHA={CONF_BASE_ALPHA}\n\n")
        f.write(dev_metrics["cls_report"])

    save_confusion_matrix(dev_metrics["cm"],  "dev",  rare_labels)
    save_per_class_f1_chart(dev_metrics["per_class_metrics"], "dev", rare_labels)

    print("\nEvaluating on Test set ...")
    test_metrics = kg_trainer.evaluate(test_dataset, rare_ids,
                                        split_name="test",
                                        measure_inference_time=True)
    print(f"  Test Accuracy:{test_metrics['accuracy']:.4f} "
          f"Macro-F1:{test_metrics['macro_f1']:.4f} "
          f"Rare-F1:{test_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "test_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT+BiLSTM+MHA+CRF+DualKG-RAG+AdaptiveEdge+SCL+Mixup\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(test_metrics["cls_report"])

    save_confusion_matrix(test_metrics["cm"], "test", rare_labels)
    save_per_class_f1_chart(test_metrics["per_class_metrics"], "test", rare_labels)

    pd.DataFrame({
        "true": [id2label[x] for x in test_metrics["all_trues"]],
        "pred": [id2label[x] for x in test_metrics["all_preds"]],
    }).to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    # ── JSON summary ──────────────────────────────────────
    scalar_keys = [
        "macro_f1", "micro_f1", "weighted_f1", "rare_f1",
        "macro_precision", "micro_precision", "weighted_precision", "rare_precision",
        "macro_recall",    "micro_recall",    "weighted_recall",    "rare_recall",
        "accuracy",
    ]
    metrics_summary = {
        "model": "InLegalBERT+BiLSTM+MHA+CRF+DualKG-RAG+AdaptiveEdge+SCL+Mixup",
        "new_features": {
            "scl": [
                "SupConLoss — per-class inv-freq weights + minority boost",
                "BalancedContrastiveSampler — minority oversampling w/ Gaussian noise",
                "scl_proj head on sent_vecs (Phase A)",
                "fused_scl_proj on fused_sent_vecs (Phase B)",
            ],
            "mixup": [
                "S1 IntraRare GPU ManifoldMixup — densifies rare embedding manifold",
                "S2 KG-Guided Mixup — mix rare query with top-1 DualKG neighbour",
                "S3 Rare↔HardMaj Interclass Mixup — sharpens decision boundary",
                "SoftLabelCrossEntropy (temperature=0.5) for all Mixup targets",
            ],
            "supporting": [
                "FocalLoss (gamma_rare=2.5, gamma_maj=1.0) + inv-freq class weights",
                "WeightedRandomSampler (rare oversample ratio=3.0)",
                "HardExampleBuffer (size=300) + replay in Phase B",
            ],
            "dual_kg_adaptive": [
                "compute_confusion_pairs_and_matrix() — base model confusion matrix",
                "compute_adaptive_confusion_weights() — W[i,j]=α+(1-α)*row_norm(C)[i,j]",
                "KnowledgeGraph.add_confusion_edges() — adaptive cross-edges",
                "DualKnowledgeGraph.build_confusion_edges() — G_all + G_min",
                "MinorityAwareRetriever.retrieve() — walks conf_cx_edges",
            ],
        },
        "config": {
            "scl_weight": SCL_WEIGHT, "scl_temperature": SCL_TEMPERATURE,
            "scl_minority_boost": SCL_MINORITY_BOOST,
            "mixup_alpha": MIXUP_ALPHA, "mixup_loss_weight": MIXUP_LOSS_WEIGHT,
            "kg_mixup_alpha": KG_MIXUP_ALPHA, "kg_mixup_weight": KG_MIXUP_WEIGHT,
            "inter_mixup_alpha": INTER_MIXUP_ALPHA,
            "intra_rare_weight": INTRA_RARE_MIXUP_WEIGHT,
            "conf_base_alpha": CONF_BASE_ALPHA, "conf_top_k": CONF_TOP_K,
            "focal_gamma_rare": FOCAL_GAMMA_RARE, "focal_gamma_maj": FOCAL_GAMMA_MAJ,
            "oversample_ratio": OVERSAMPLE_RARE_RATIO,
            "g_all_conf_edges": len(dual_kg.g_all.conf_cx_edges),
            "g_min_conf_edges": len(dual_kg.g_min.conf_cx_edges),
        },
        "timing": {
            "phase_a_s": base_time, "phase_b_s": kg_time,
            "total_s": base_time + kg_time,
        },
        "rare_classes": rare_labels,
        "dev":  {k: dev_metrics[k]  for k in scalar_keys},
        "test": {k: test_metrics[k] for k in scalar_keys},
        "per_class_dev":  dev_metrics["per_class_metrics"],
        "per_class_test": test_metrics["per_class_metrics"],
    }
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(metrics_summary, f, indent=2)

    print_metrics_table(dev_metrics, test_metrics,
                        base_time=base_time, kg_time=kg_time,
                        total_trainable=total_trainable)
    print(f"\n📁 All outputs saved to: {OUT_DIR}/")


if __name__ == "__main__":
    main()

Device : cuda:0
Architecture : InLegalBERT + BiLSTM + MHA + CRF
               → Dual KG-RAG (G_all + G_min) + Adaptive Confusion Edges
               + Supervised Contrastive Learning (SCL)
               + GPU Manifold Mixup (S1 IntraRare | S2 KG-Guided | S3 Inter-class)
               + Focal Loss  +  Rare Oversampling  +  Hard Replay

Loading JSONL files ...
  Train:245 | Dev:30 | Test:50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167 samples)
   FAC                  19.99%  ( 5744 samples)
   RLC                   2.62%  (  752 samples) ← RARE
   ISSUE                 1.28%  (  367 samples) ← RARE
   ARG_PETITIONER        4.58%  ( 1315 samples) ← RARE
   ARG_RESPONDENT        2.43%  (  698 samples) ← RARE
   ANALYSIS             36.66%  (10537 samples)
   STA                   1.67%  (  481 samples) ← RARE
   PRE_RELIED            4.97%  ( 1427 samples) ← RARE
   PRE_NOT_RELIED        0.55%  (  158 samples) ← RARE
   RATIO                 2.30

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



❄️  BERT frozen: embeddings + layers 0-7.
🔥 BERT trainable: layers 8-11 + pooler.

[Base+SCL] Ep 001/60 | loss:289.0829 val_loss:211.3799 | mac_F1:0.0391 rare_F1:0.0000 | t:97.9s ES:0/10
  ✔ New best val_macro_f1=0.0391
[Base+SCL] Ep 002/60 | loss:230.0271 val_loss:164.2266 | mac_F1:0.1096 rare_F1:0.0000 | t:147.1s ES:0/10
  ✔ New best val_macro_f1=0.1096
[Base+SCL] Ep 003/60 | loss:171.2669 val_loss:128.7104 | mac_F1:0.2185 rare_F1:0.0736 | t:85.6s ES:0/10
  ✔ New best val_macro_f1=0.2185
[Base+SCL] Ep 004/60 | loss:154.7798 val_loss:96.8936 | mac_F1:0.2790 rare_F1:0.1140 | t:94.1s ES:0/10
  ✔ New best val_macro_f1=0.2790
[Base+SCL] Ep 005/60 | loss:119.0183 val_loss:89.1943 | mac_F1:0.2743 rare_F1:0.1067 | t:94.7s ES:0/10
[Base+SCL] Ep 006/60 | loss:113.6544 val_loss:83.8670 | mac_F1:0.2918 rare_F1:0.1290 | t:100.6s ES:1/10
  ✔ New best val_macro_f1=0.2918
[Base+SCL] Ep 007/60 | loss:106.5705 val_loss:79.0289 | mac_F1:0.3287 rare_F1:0.1750 | t:87.1s ES:0/10
  ✔ New best val_macro_f1